In [1]:
import os
import numpy as np
import rasterio
from tqdm import tqdm

input_dir = '/home/ubuntu/work/saved_data/landsat_disturbance_detection/1D_U_Net/Multikernel_Model/predictions'
output_file = '/home/ubuntu/work/saved_data/landsat_disturbance_detection/1D_U_Net/Multikernel_Model/final_yearly_predictions_sequential_fast.tif'

# Dictionary to store predictions by year
yearly_predictions = {}

# Process files in sorted order
prediction_files = sorted([f for f in os.listdir(input_dir) if f.endswith('.tif')])
if not prediction_files:
    raise ValueError("No prediction files found in directory")

# Process the first file - take all channels
first_file = prediction_files[0]
first_file_path = os.path.join(input_dir, first_file)
print(f"\nProcessing first file: {first_file}")

with rasterio.open(first_file_path) as src:
    # Save metadata for later use
    meta = src.meta.copy()
    height, width = src.height, src.width
    transform = src.transform
    crs = src.crs
    dtype = src.dtypes[0]
    
    # Read all bands from first file
    for band_idx in range(1, src.count + 1):
        band_data = src.read(band_idx)
        year = int(src.descriptions[band_idx-1].split()[-1])
        yearly_predictions[year] = band_data
        print(f"First file - Band {band_idx}: Year {year}")

# Process remaining files - take only last channel
for fname in tqdm(prediction_files[1:], desc="Processing remaining files"):
    path = os.path.join(input_dir, fname)
    try:
        with rasterio.open(path) as src:
            # Read only the last band (most recent year in the window)
            last_band_idx = src.count
            last_band_data = src.read(last_band_idx)
            
            # Get the year this band represents from its description
            year_from_desc = int(src.descriptions[last_band_idx-1].split()[-1])
            
            # Store the prediction
            yearly_predictions[year_from_desc] = last_band_data
            
            print(f"Processed {fname} - Last band represents year {year_from_desc}")
            
    except Exception as e:
        print(f"Error processing {fname}: {e}")
        continue

# Sort years and create final stack
sorted_years = sorted(yearly_predictions.keys())
stacked_predictions = []

print("\nCollecting predictions for years:")
for year in sorted_years:
    prediction = yearly_predictions[year]
    stacked_predictions.append(prediction)
    print(f"Year {year}: Shape {prediction.shape}, Type {prediction.dtype}")
    
    # Verify data
    if not np.isfinite(prediction).all():
        print(f"Warning: Found non-finite values in year {year}, replacing with 0")
        prediction = np.nan_to_num(prediction, nan=0, posinf=0, neginf=0)
    if prediction.dtype != np.uint8:
        print(f"Warning: Converting data for year {year} from {prediction.dtype} to uint8")
        prediction = prediction.astype(np.uint8)

# Update metadata for output file
meta.update({
    "count": len(sorted_years),
    "dtype": rasterio.uint8,
    "compress": "lzw",  # Simple, widely supported compression
    "driver": "GTiff",
    "tiled": True,
    "blockxsize": 256,
    "blockysize": 256,
    "interleave": "band",  # Required setting
    "bigtiff": "YES",  # Support for large files
    "predictor": 2,  # Horizontal differencing for better compression
    "zlevel": 1  # Light compression for better compatibility
})

try:
    # Write with error handling
    with rasterio.open(output_file, "w", **meta) as dst:
        for i, year in enumerate(sorted_years):
            try:
                # Write band
                dst.write(stacked_predictions[i], i + 1)
                dst.set_band_description(i + 1, f"Prediction for year {year}")
            except Exception as e:
                print(f"Error writing band {i+1} (year {year}): {e}")
                raise

    # Verify the file was written correctly
    with rasterio.open(output_file) as src:
        print("\nVerifying output file:")
        print(f"Number of bands: {src.count}")
        print(f"File size: {os.path.getsize(output_file) / (1024*1024):.1f} MB")
        
        # Read a small test block from each band
        for i in range(src.count):
            test_data = src.read(i + 1, window=((0, 256), (0, 256)))
            print(f"Band {i+1} test block shape: {test_data.shape}, dtype: {test_data.dtype}")
            year = sorted_years[i]
            print(f"Band {i+1}: Year {year}")

    print(f"\nSuccessfully saved and verified final predictions to {output_file}")

except Exception as e:
    print(f"\nError saving predictions: {e}")
    print("Trying alternative save method...")
    
    # Try alternative save method with minimal settings
    basic_meta = {
        "driver": "GTiff",
        "height": height,
        "width": width,
        "count": len(sorted_years),
        "dtype": rasterio.uint8,
        "crs": crs,
        "transform": transform,
        "interleave": "band",  # Required setting
        "bigtiff": "YES",  # Support for large files
        "tiled": True,
        "blockxsize": 256,
        "blockysize": 256
    }
    
    with rasterio.open(output_file, "w", **basic_meta) as dst:
        for i, year in enumerate(sorted_years):
            dst.write(stacked_predictions[i], i + 1)
            dst.set_band_description(i + 1, f"Prediction for year {year}")
    
    print(f"Saved with basic settings to {output_file}")

# Final verification
try:
    with rasterio.open(output_file) as src:
        print("\nFinal verification:")
        print(f"File exists: {os.path.exists(output_file)}")
        print(f"File size: {os.path.getsize(output_file) / (1024*1024):.1f} MB")
        print(f"Number of bands: {src.count}")
        print(f"Data type: {src.dtypes[0]}")
        print(f"CRS: {src.crs}")
        print(f"Transform: {src.transform}")
        
        # Print year for each band
        print("\nBand to year mapping:")
        for i in range(src.count):
            year = sorted_years[i]
            print(f"Band {i+1}: Year {year}")
        
        # Test read a small block from first and last band
        first_block = src.read(1, window=((0, 256), (0, 256)))
        last_block = src.read(src.count, window=((0, 256), (0, 256)))
        print(f"\nFirst band test block shape: {first_block.shape}")
        print(f"Last band test block shape: {last_block.shape}")
except Exception as e:
    print(f"\nError in final verification: {e}")


Processing first file: 1991_disturbed_undisturbed_pred_unet_lansatbands_w8.tif
First file - Band 1: Year 1984
First file - Band 2: Year 1985
First file - Band 3: Year 1986
First file - Band 4: Year 1987
First file - Band 5: Year 1988
First file - Band 6: Year 1989
First file - Band 7: Year 1990
First file - Band 8: Year 1991


Processing remaining files:   3%|▎         | 1/32 [00:00<00:17,  1.78it/s]

Processed 1992_disturbed_undisturbed_pred_unet_lansatbands_w8.tif - Last band represents year 1992


Processing remaining files:   6%|▋         | 2/32 [00:01<00:16,  1.81it/s]

Processed 1993_disturbed_undisturbed_pred_unet_lansatbands_w8.tif - Last band represents year 1993


...

Processing remaining files:  97%|█████████▋| 31/32 [00:16<00:00,  1.84it/s]

Processed 2022_disturbed_undisturbed_pred_unet_lansatbands_w8.tif - Last band represents year 2022


Processing remaining files: 100%|██████████| 32/32 [00:17<00:00,  1.88it/s]

Processed 2023_disturbed_undisturbed_pred_unet_lansatbands_w8.tif - Last band represents year 2023

Year 1984: Shape (5000, 5000), Type uint8
Year 1985: Shape (5000, 5000), Type uint8
Year 1986: Shape (5000, 5000), Type uint8
Year 1987: Shape (5000, 5000), Type uint8
Year 1988: Shape (5000, 5000), Type uint8
Year 1989: Shape (5000, 5000), Type uint8
Year 1990: Shape (5000, 5000), Type uint8
Year 1991: Shape (5000, 5000), Type uint8
Year 1992: Shape (5000, 5000), Type uint8
Year 1993: Shape (5000, 5000), Type uint8
Year 1994: Shape (5000, 5000), Type uint8
Year 1995: Shape (5000, 5000), Type uint8
Year 1996: Shape (5000, 5000), Type uint8
Year 1997: Shape (5000, 5000), Type uint8
Year 1998: Shape (5000, 5000), Type uint8
Year 1999: Shape (5000, 5000), Type uint8
Year 2000: Shape (5000, 5000), Type uint8
Year 2001: Shape (5000, 5000), Type uint8
Year 2002: Shape (5000, 5000), Type uint8
Year 2003: Shape (5000, 5000), Type uint8
Year 2004: Shape (5000, 5000), Type uint8
Year 2005: Shape (


Verifying output file:
Number of bands: 40
File size: 32.4 MB
Band 1 test block shape: (256, 256), dtype: uint8
Band 1: Year 1984
Band 2 test block shape: (256, 256), dtype: uint8
Band 2: Year 1985
Band 3 test block shape: (256, 256), dtype: uint8
Band 3: Year 1986
Band 4 test block shape: (256, 256), dtype: uint8
Band 4: Year 1987
Band 5 test block shape: (256, 256), dtype: uint8
Band 5: Year 1988
Band 6 test block shape: (256, 256), dtype: uint8
Band 6: Year 1989
Band 7 test block shape: (256, 256), dtype: uint8
Band 7: Year 1990
Band 8 test block shape: (256, 256), dtype: uint8
Band 8: Year 1991
Band 9 test block shape: (256, 256), dtype: uint8
Band 9: Year 1992
Band 10 test block shape: (256, 256), dtype: uint8
Band 10: Year 1993
Band 11 test block shape: (256, 256), dtype: uint8
Band 11: Year 1994
Band 12 test block shape: (256, 256), dtype: uint8
Band 12: Year 1995
Band 13 test block shape: (256, 256), dtype: uint8
Band 13: Year 1996
Band 14 test block shape: (256, 256), dtype: 